# News-Augmented Hourly Electricity Price Forecasting for Germany

**Version:** 1.0  
**Date:** 2025-10-11

This notebook implements a machine learning model for forecasting day-ahead hourly electricity prices in Germany, incorporating German news data to improve predictive accuracy.


## Setup: Import Libraries


In [1]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import time
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

# API imports
from newsapi import NewsApiClient

# Environment variables
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Try to import XGBoost
try:
    from xgboost import XGBRegressor
    xgboost_available = True
    print("✓ XGBoost imported successfully")
except ImportError:
    xgboost_available = False
    print("✗ XGBoost not found. Installing...")
    print("Please run: pip install xgboost")
    print("Or in your notebook: !pip install xgboost")

# NLP
try:
    from sentence_transformers import SentenceTransformer
    sentence_transformers_available = True
    print("✓ sentence-transformers imported successfully")
except ImportError:
    sentence_transformers_available = False
    print("✗ sentence-transformers not found. Installing...")
    print("Please run: pip install sentence-transformers")
    print("Or in your notebook: !pip install sentence-transformers")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("colorblind")

print("\n" + "="*60)
print("IMPORT SUMMARY")
print("="*60)
print("✓ Core libraries imported successfully")
print(f"✓ Environment variables loaded from .env file")
if not xgboost_available or not sentence_transformers_available:
    print("\n⚠️  Missing packages detected - see installation instructions above")
print("="*60)


✓ XGBoost imported successfully
✓ sentence-transformers imported successfully

IMPORT SUMMARY
✓ Core libraries imported successfully
✓ Environment variables loaded from .env file


## Module 1: Data Acquisition

### 1.1: Fetch Energy Data from Energy Charts API


In [2]:
def fetch_energy_charts_data(start_date: str, end_date: str, bidding_zone: str = 'DE-LU') -> pd.DataFrame:
    """
    Fetch hourly price and power generation data from Energy Charts API.
    
    Parameters:
    -----------
    start_date : str
        Start date in format 'YYYY-MM-DD'
    end_date : str
        End date in format 'YYYY-MM-DD'
    bidding_zone : str
        Bidding zone code (default: 'DE-LU' for Germany/Luxembourg)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with datetime index and columns: 'price', 'total_power'
    """
    base_url = "https://api.energy-charts.info"
    
    # Fetch price data (day-ahead spot price)
    print(f"Fetching price data from {start_date} to {end_date} for bidding zone {bidding_zone}...")
    price_url = f"{base_url}/price"
    price_params = {
        'bzn': bidding_zone,  # Bidding zone (DE-LU for Germany/Luxembourg)
        'start': start_date,
        'end': end_date
    }
    
    try:
        price_response = requests.get(price_url, params=price_params, timeout=30)
        price_response.raise_for_status()
        price_data = price_response.json()
        print(f"Price data received successfully")
        print(f"Response keys: {list(price_data.keys())}")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching price data: {e}")
        print(f"URL: {price_url}")
        print(f"Params: {price_params}")
        raise
    
    # Fetch total power generation data
    print(f"Fetching power generation data from {start_date} to {end_date}...")
    power_url = f"{base_url}/total_power"
    power_params = {
        'country': 'de',  # Only available for Germany according to docs
        'start': start_date,
        'end': end_date
    }
    
    try:
        power_response = requests.get(power_url, params=power_params, timeout=30)
        power_response.raise_for_status()
        power_data = power_response.json()
        print(f"Power data received successfully")
        print(f"Response keys: {list(power_data.keys())}")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching power data: {e}")
        print(f"URL: {power_url}")
        print(f"Params: {power_params}")
        raise
    
    # Parse price data
    if 'unix_seconds' in price_data and 'price' in price_data:
        price_timestamps = pd.to_datetime(price_data['unix_seconds'], unit='s')
        prices = price_data['price']
        price_df = pd.DataFrame({'price': prices}, index=price_timestamps)
        print(f"Price data parsed: {len(price_df)} records")
    else:
        print(f"Available keys in price_data: {price_data.keys()}")
        raise ValueError("Unexpected price data structure")
    
    # Parse power data - aggregate all production types
    if 'unix_seconds' in power_data and 'production_types' in power_data:
        power_timestamps = pd.to_datetime(power_data['unix_seconds'], unit='s')
        print(f"Power timestamps: {len(power_timestamps)} records")
        print(f"Production types: {len(power_data['production_types'])} types")
        
        # Sum all production types to get total power
        total_power = np.zeros(len(power_timestamps))
        for i, production_type in enumerate(power_data['production_types']):
            print(f"Processing production type {i+1}: {production_type['name']} ({len(production_type['data'])} values)")
            try:
                production_values = np.array(production_type['data'])
                total_power += production_values
            except Exception as e:
                print(f"Error processing production type {production_type['name']}: {e}")
                continue
        
        power_df = pd.DataFrame({'total_power': total_power}, index=power_timestamps)
        print(f"Power data parsed: {len(power_df)} records")
    else:
        print(f"Available keys in power_data: {power_data.keys()}")
        raise ValueError("Unexpected power data structure")
    
    # Merge the two datasets
    energy_df = price_df.join(power_df, how='outer')
    
    print(f"Energy data fetched: {len(energy_df)} records")
    print(f"Date range: {energy_df.index.min()} to {energy_df.index.max()}")
    
    return energy_df

# Fetch 1 year of data
start_date = '2023-10-01'
end_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')

energy_df = fetch_energy_charts_data(start_date, end_date)
energy_df.head()


Fetching price data from 2023-10-01 to 2025-10-10 for bidding zone DE-LU...
Price data received successfully
Response keys: ['license_info', 'unix_seconds', 'price', 'unit', 'deprecated']
Fetching power generation data from 2023-10-01 to 2025-10-10...
Power data received successfully
Response keys: ['unix_seconds', 'production_types', 'deprecated']
Price data parsed: 18504 records
Power timestamps: 71136 records
Production types: 22 types
Processing production type 1: Hydro pumped storage consumption (71136 values)
Processing production type 2: Cross border electricity trading (71136 values)
Processing production type 3: Nuclear (71136 values)
Error processing production type Nuclear: Cannot cast ufunc 'add' output from dtype('O') to dtype('float64') with casting rule 'same_kind'
Processing production type 4: Hydro Run-of-River (71136 values)
Processing production type 5: Biomass (71136 values)
Processing production type 6: Fossil brown coal / lignite (71136 values)
Processing producti

,price,total_power
2023-09-30 22:00:00,102.73,116105.5
2023-09-30 22:15:00,NaN,113632.9
2023-09-30 22:30:00,NaN,111751.8
2023-09-30 22:45:00,NaN,110998.5
2023-09-30 23:00:00,94.14,110498.5


### 1.2: Fetch News Data from NewsAPI


In [3]:
def fetch_news_data(start_date: str, end_date: str, api_key: str) -> pd.DataFrame:
    """
    Fetch German news articles from NewsAPI.org.
    
    Parameters:
    -----------
    start_date : str
        Start date in format 'YYYY-MM-DD'
    end_date : str
        End date in format 'YYYY-MM-DD'
    api_key : str
        NewsAPI.org API key
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: 'publishedAt', 'title', 'description', 'source'
    """
    # Initialize NewsAPI.org client
    newsapi = NewsApiClient(api_key=api_key)
    
    # German news sources as specified in PRD
    SOURCES = 'bild, der-tagesspiegel,die-zeit,focus,handelsblatt,spiegel-online,wirtschafts-woche'
    
    # Define search keywords - FOCUSED ON ELECTRICITY CONSUMPTION & DEMAND
    # Refined to capture news that impacts electricity consumption patterns
    ENERGY_KEYWORDS = (
        # Direct consumption/demand terms
        '"Stromverbrauch" OR "Energieverbrauch" OR "Strombedarf" OR '
        '"Stromnachfrage" OR "Netzlast" OR "Lastspitzen" OR '
        # Price-related (impacts consumption behavior)
        '"Strompreis" OR "Energiepreis" OR "Stromkosten" OR '
        # Consumer/household impact (behavioral changes)
        '"Stromrechnung" OR "Energiekosten Haushalt" OR '
        # Crisis/supply issues (affect consumption)
        '"Energiekrise" OR "Stromausfall" OR "Energieversorgung" OR '
        # Weather-related (impacts demand)
        '"Heizung" OR "Klimaanlage" OR "Kältewelle" OR "Hitzewelle"'
    )
    
    print(f"Fetching articles from NewsAPI.org in weekly (7-day) increments...")
    print(f"Keywords (CONSUMPTION-FOCUSED): {ENERGY_KEYWORDS}")
    print(f"Sources: {SOURCES}")
    
    try:
        # Calculate date ranges - fetch in weekly increments (pro key allows full historical access)
        today = datetime.strptime(end_date, '%Y-%m-%d')
        start = datetime.strptime(start_date, '%Y-%m-%d')
        all_articles = []
        
        # Calculate number of weekly periods needed
        total_days = (today - start).days
        num_periods = (total_days // 7) + (1 if total_days % 7 > 0 else 0)
        
        for period in range(num_periods):
            # Calculate 'to' date (end of weekly period)
            to_date = today - timedelta(days=period * 7)
            # Calculate 'from' date (start of weekly period)
            from_date = to_date - timedelta(days=7)
            
            # Format dates for API (ISO 8601 format)
            from_str = from_date.strftime('%Y-%m-%d')
            to_str = to_date.strftime('%Y-%m-%d')
            
            print(f"\nPeriod {period + 1}/{num_periods}: {from_str} to {to_str}")
            
            # Fetch articles for this period
            articles_period = newsapi.get_everything(
                q=ENERGY_KEYWORDS,
                sources=SOURCES,
                language='de',
                sort_by='publishedAt',
                from_param=from_str,
                to=to_str,
                page_size=100
            )
            
            print(f"  Total results available: {articles_period['totalResults']}")
            print(f"  Articles retrieved: {len(articles_period['articles'])}")
            
            # Add articles to list
            all_articles.extend(articles_period['articles'])
        
        # Convert all articles to DataFrame
        if not all_articles:
            print("Warning: No articles fetched!")
            return pd.DataFrame(columns=['publishedAt', 'title', 'description', 'source'])
        
        news_df = pd.DataFrame([
            {
                'publishedAt': article['publishedAt'],
                'title': article.get('title', ''),
                'description': article.get('description', ''),
                'source': article['source']['name']
            }
            for article in all_articles
        ])
        
        print(f"\n✓ Initial articles collected: {len(news_df)}")
        
        # Enhanced duplicate removal with detailed logging
        if not news_df.empty and 'title' in news_df.columns:
            initial_count = len(news_df)
            
            # Check for completely empty or null titles (these should be removed)
            empty_titles = news_df['title'].isna() | (news_df['title'].str.strip() == '')
            if empty_titles.any():
                print(f"  ⚠️  Found {empty_titles.sum()} articles with empty titles (will be removed)")
                news_df = news_df[~empty_titles]
            
            # Remove exact title duplicates (keeps first occurrence by date)
            # Sort by publishedAt first to ensure we keep the earliest article
            news_df = news_df.sort_values('publishedAt')
            
            # Show some examples of what will be deduplicated
            duplicates_mask = news_df.duplicated(subset=['title'], keep='first')
            if duplicates_mask.any():
                num_duplicates = duplicates_mask.sum()
                print(f"\n  📊 Duplicate Analysis:")
                print(f"     - Found {num_duplicates} exact title duplicates")
                print(f"     - These are likely republished/updated articles across time periods")
                
                # Show a few examples
                duplicate_titles = news_df[duplicates_mask]['title'].head(3).tolist()
                if duplicate_titles:
                    print(f"     - Example duplicates:")
                    for i, title in enumerate(duplicate_titles, 1):
                        print(f"       {i}. {title[:80]}...")
            
            # Perform deduplication
            news_df = news_df.drop_duplicates(subset=['title'], keep='first')
            duplicates_removed = initial_count - len(news_df)
            
            if duplicates_removed > 0:
                print(f"\n  ✓ Removed {duplicates_removed} duplicate/empty articles")
                print(f"  ✓ Kept {len(news_df)} unique articles (earliest occurrence of each)")
            else:
                print(f"  ✓ No duplicates found - all {len(news_df)} articles are unique")
        
        # Parse dates
        news_df['publishedAt'] = pd.to_datetime(news_df['publishedAt'])
        
        print(f"\n{'='*60}")
        print(f"✓ DataFrame created with shape: {news_df.shape}")
        print(f"✓ Total unique articles retrieved: {len(news_df)}")
        print(f"✓ Date range: {news_df['publishedAt'].min()} to {news_df['publishedAt'].max()}")
        print(f"{'='*60}")
        
        return news_df
        
    except Exception as e:
        print(f"✗ Error: {e}")
        return pd.DataFrame(columns=['publishedAt', 'title', 'description', 'source'])

# Load NewsAPI key from environment variable
NEWS_API_KEY = os.getenv('NEWSAPIORG_KEY')

if NEWS_API_KEY:
    print(f"NewsAPI key loaded successfully from .env file")
    # Fetch real news data
    news_df = fetch_news_data(start_date, end_date, NEWS_API_KEY)
    news_df.head()
else:
    print("Warning: NEWSAPIORG_KEY not found in .env file")
    print("Please ensure your .env file contains: NEWSAPIORG_KEY=your_api_key_here")


NewsAPI key loaded successfully from .env file
Fetching articles from NewsAPI.org in weekly (7-day) increments...
Keywords (CONSUMPTION-FOCUSED): "Stromverbrauch" OR "Energieverbrauch" OR "Strombedarf" OR "Stromnachfrage" OR "Netzlast" OR "Lastspitzen" OR "Strompreis" OR "Energiepreis" OR "Stromkosten" OR "Stromrechnung" OR "Energiekosten Haushalt" OR "Energiekrise" OR "Stromausfall" OR "Energieversorgung" OR "Heizung" OR "Klimaanlage" OR "Kältewelle" OR "Hitzewelle"
Sources: bild, der-tagesspiegel,die-zeit,focus,handelsblatt,spiegel-online,wirtschafts-woche

Period 1/106: 2025-10-03 to 2025-10-10
  Total results available: 77
  Articles retrieved: 77

Period 2/106: 2025-09-26 to 2025-10-03
  Total results available: 92
  Articles retrieved: 92

Period 3/106: 2025-09-19 to 2025-09-26
  Total results available: 80
  Articles retrieved: 78

Period 4/106: 2025-09-12 to 2025-09-19
  Total results available: 104
  Articles retrieved: 100

Period 5/106: 2025-09-05 to 2025-09-12
✗ Error: {'st

In [4]:
# Visualize article counts by source and publication time distribution
source_counts = news_df['source'].value_counts().head(5)

# Extract hour of publication
news_df['publication_hour'] = news_df['publishedAt'].dt.hour
hour_counts = news_df['publication_hour'].value_counts().sort_index()

# Create figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Article count by source
source_counts.plot(kind='bar', ax=axes[0], color='#1f77b4', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Source')
axes[0].set_ylabel('Number of Articles')
axes[0].set_title('Article Count by Source (Top 5)')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

# Subplot 2: Distribution by hour of publication
axes[1].bar(hour_counts.index, hour_counts.values, color='#ff7f0e', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Hour of Day (UTC)')
axes[1].set_ylabel('Number of Articles')
axes[1].set_title('Article Distribution by Publication Hour')
axes[1].set_xticks(range(0, 24, 2))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nTop 5 sources:")
for source, count in source_counts.items():
    print(f"  {source}: {count} articles")

print(f"\nPublication hour statistics:")
print(f"  Most active hour: {hour_counts.idxmax()}:00 ({hour_counts.max()} articles)")
print(f"  Least active hour: {hour_counts.idxmin()}:00 ({hour_counts.min()} articles)")
print(f"  Average articles per hour: {hour_counts.mean():.1f}")
print(f"  Peak hours (>20 articles): {list(hour_counts[hour_counts > 20].index)}")


AttributeError: Can only use .dt accessor with datetimelike values

## Module 2: Data Preprocessing & Alignment

### 2.1: Preprocess Energy Data


In [ ]:
# Check data types and missing values
print("Energy DataFrame Info:")
print(energy_df.info())
print(f"\nMissing values:\n{energy_df.isnull().sum()}")

# Handle missing values using interpolation
energy_df = energy_df.interpolate(method='time').ffill().bfill()

# Ensure correct data types
energy_df['price'] = energy_df['price'].astype(float)
energy_df['total_power'] = energy_df['total_power'].astype(float)

print(f"\nAfter preprocessing - Missing values:\n{energy_df.isnull().sum()}")
energy_df.describe()


### 2.2: Preprocess News Data

Aggregate all news articles per day into a single text document.


In [ ]:
def preprocess_news_data(news_df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare individual news articles for time-decayed embedding.
    NO AGGREGATION - keep each article separate with its timestamp.
    
    Parameters:
    -----------
    news_df : pd.DataFrame
        Raw news DataFrame with 'publishedAt', 'title', 'description'
    
    Returns:
    --------
    pd.DataFrame
        Individual articles with columns: 'publishedAt', 'article_text'
    """
    # Combine title and description, handle NaN values
    news_df['article_text'] = (
        news_df['title'].fillna('') + '. ' + news_df['description'].fillna('')
    ).str.strip()
    
    # Remove any empty text
    news_df = news_df[news_df['article_text'].str.len() > 0].copy()
    
    # Keep only what we need: timestamp and text
    processed_news_df = news_df[['publishedAt', 'article_text']].copy()
    
    # Sort by publication time
    processed_news_df = processed_news_df.sort_values('publishedAt')
    
    print(f"✓ Individual articles prepared: {len(processed_news_df)} articles")
    print(f"  Date range: {processed_news_df['publishedAt'].min()} to {processed_news_df['publishedAt'].max()}")
    
    return processed_news_df

# Process the news data if it was fetched successfully
if 'news_df' in locals() and len(news_df) > 0:
    processed_news_df = preprocess_news_data(news_df)
    print(f"\nSample of processed news:")
    display(processed_news_df.head())
else:
    # Placeholder for development if news fetch failed
    print("Using placeholder news data for development purposes.")
    dates = pd.date_range(start=start_date, end=end_date, freq='H')
    processed_news_df = pd.DataFrame({
        'publishedAt': dates,
        'article_text': ['Sample news text for ' + str(d) for d in dates]
    })
    print(f"Created placeholder news data: {len(processed_news_df)} articles")


### 2.3: Prepare Energy Data for Time-Decayed News Integration

We'll create embeddings for each individual article, then compute time-weighted averages for each hour.


In [ ]:
# Start with energy_df as our base (will add news features later)
master_df = energy_df.copy()

print(f"Energy DataFrame prepared: {len(master_df)} records")
print(f"Date range: {master_df.index.min()} to {master_df.index.max()}")
print(f"\nColumns: {master_df.columns.tolist()}")
print("\nNews features will be added after embedding...")
master_df.head()


## Module 3: Feature Engineering

### 3.1: Create Baseline Features

Temporal features, lag features, and rolling window features.


In [ ]:
# Temporal features
master_df['hour'] = master_df.index.hour
master_df['day_of_week'] = master_df.index.dayofweek
master_df['day_of_year'] = master_df.index.dayofyear
master_df['month'] = master_df.index.month
master_df['week_of_year'] = master_df.index.isocalendar().week.values

# Lag features for price
master_df['price_lag_24h'] = master_df['price'].shift(24)
master_df['price_lag_168h'] = master_df['price'].shift(168)  # 7 days = 168 hours

# Lag features for power
master_df['power_lag_24h'] = master_df['total_power'].shift(24)
master_df['power_lag_168h'] = master_df['total_power'].shift(168)

# Rolling window features for price
master_df['price_rolling_mean_24h'] = master_df['price'].rolling(window=24, min_periods=1).mean()
master_df['price_rolling_std_24h'] = master_df['price'].rolling(window=24, min_periods=1).std()

# Rolling window features for power
master_df['power_rolling_mean_24h'] = master_df['total_power'].rolling(window=24, min_periods=1).mean()

# Drop rows with NaN values from lag/rolling features
master_df = master_df.dropna()

print(f"Baseline features created. Rows after dropping NaN: {len(master_df)}")
print(f"\nFeatures: {[col for col in master_df.columns if col != 'news_feature_text']}")
master_df.describe()


### 3.2: Create Time-Decayed News Embeddings

**NEW APPROACH:**
1. Embed each article individually (no aggregation)
2. For each hour H, compute weighted average of recent news embeddings
3. Weight = exp(-λ * hours_since_publication) - exponential time decay
4. This gives each hour a unique embedding based on news recency


In [ ]:
# STEP 1: Load embedding model and embed each article individually
print("="*70)
print("STEP 1: Embedding Individual News Articles")
print("="*70)

print("Loading German-compatible BERT model...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f"Encoding {len(processed_news_df)} individual articles...")
article_embeddings = model.encode(
    processed_news_df['article_text'].tolist(), 
    show_progress_bar=True, 
    batch_size=32
)

print(f"✓ Article embeddings created: {article_embeddings.shape}")
print(f"  Embedding dimension: {article_embeddings.shape[1]}")

# Store embeddings with their timestamps
processed_news_df['embedding'] = list(article_embeddings)

# STEP 2: Define time-decay function
print("\n" + "="*70)
print("STEP 2: Computing Time-Decayed Embeddings for Each Hour")
print("="*70)

def compute_time_decayed_embedding(target_time, news_df, decay_lambda=0.1, lookback_hours=168):
    """
    Compute time-weighted average of news embeddings for a specific hour.
    
    Parameters:
    -----------
    target_time : datetime
        The hour we're predicting for
    news_df : DataFrame
        News articles with 'publishedAt' and 'embedding' columns
    decay_lambda : float
        Decay rate (higher = faster decay). Default 0.1 means ~37% weight after 10 hours
    lookback_hours : int
        Maximum hours to look back for news (default: 168 = 1 week)
    
    Returns:
    --------
    np.array
        Time-weighted average embedding
    """
    # Ensure target_time is timezone-aware (match news timestamps)
    if target_time.tzinfo is None:
        target_time = target_time.tz_localize('UTC')
    
    # Filter to news published BEFORE the target time (causal constraint)
    # and within the lookback window
    time_window_start = target_time - pd.Timedelta(hours=lookback_hours)
    relevant_news = news_df[
        (news_df['publishedAt'] < target_time) & 
        (news_df['publishedAt'] >= time_window_start)
    ]
    
    if len(relevant_news) == 0:
        # No news in window - return zero embedding
        return np.zeros(384)
    
    # Calculate hours since publication
    hours_ago = (target_time - relevant_news['publishedAt']).dt.total_seconds() / 3600
    
    # Compute exponential decay weights
    weights = np.exp(-decay_lambda * hours_ago)
    
    # Normalize weights to sum to 1
    weights = weights / weights.sum()
    
    # Compute weighted average of embeddings
    embeddings_array = np.stack(relevant_news['embedding'].values)
    weighted_embedding = (embeddings_array.T @ weights).T
    
    return weighted_embedding

# STEP 3: Compute time-decayed embedding for each hour in master_df
print(f"Computing time-decayed embeddings for {len(master_df)} hours...")
print(f"Decay parameter λ = 0.1 (half-life ≈ 6.9 hours)")
print(f"Lookback window: 168 hours (7 days)")

# This will take a few minutes for large datasets
from tqdm import tqdm
time_decayed_embeddings = []

for timestamp in tqdm(master_df.index, desc="Processing hours"):
    embedding = compute_time_decayed_embedding(
        timestamp, 
        processed_news_df,
        decay_lambda=0.1,
        lookback_hours=168
    )
    time_decayed_embeddings.append(embedding)

time_decayed_embeddings = np.array(time_decayed_embeddings)
print(f"\n✓ Time-decayed embeddings computed: {time_decayed_embeddings.shape}")

# STEP 4: Apply PCA for dimensionality reduction
print("\n" + "="*70)
print("STEP 4: Dimensionality Reduction with PCA")
print("="*70)

n_components = 20  # Reduce from 384 to 20 dimensions
pca = PCA(n_components=n_components, random_state=42)
embeddings_reduced = pca.fit_transform(time_decayed_embeddings)

print(f"✓ PCA applied: {embeddings_reduced.shape}")
print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

# STEP 5: Add embeddings to master_df
print("\n" + "="*70)
print("STEP 5: Integrating News Features into Master DataFrame")
print("="*70)

embedding_columns = [f'news_embed_{i}' for i in range(n_components)]
news_embeddings_df = pd.DataFrame(
    embeddings_reduced,
    index=master_df.index,
    columns=embedding_columns
)

# Merge into master_df
master_df = master_df.join(news_embeddings_df)

print(f"\n✓ News features integrated!")
print(f"  Final master_df shape: {master_df.shape}")
print(f"  Columns: {master_df.columns.tolist()}")

# Check for any NaN values in news embeddings
nan_count = master_df[embedding_columns].isnull().sum().sum()
print(f"  NaN values in news embeddings: {nan_count}")

print("\n" + "="*70)
master_df.head()


## Module 5: Model Training

### 5.1: Define Feature Sets and Target Variable


In [ ]:
# Define target variable
y = master_df['price']

# Define baseline features (excluding news embeddings)
baseline_features = [
    'total_power',
    'hour', 'day_of_week', 'day_of_year', 'month', 'week_of_year',
    'price_lag_24h', 'price_lag_168h',
    'power_lag_24h', 'power_lag_168h',
    'price_rolling_mean_24h', 'price_rolling_std_24h',
    'power_rolling_mean_24h'
]

X_baseline = master_df[baseline_features]

# Define advanced features (baseline + news embeddings)
news_embedding_features = [col for col in master_df.columns if col.startswith('news_embed_')]
advanced_features = baseline_features + news_embedding_features

X_advanced = master_df[advanced_features]

print(f"Target variable (y): {y.shape}")
print(f"Baseline features (X_baseline): {X_baseline.shape}")
print(f"Advanced features (X_advanced): {X_advanced.shape}")
print(f"\nNumber of news embedding features: {len(news_embedding_features)}")


### 5.2: Create Chronological Train-Test Split

Use a timestamp-based split (NOT random) to respect the temporal nature of the data.


In [ ]:
# Check the actual date range of the data
print(f"Data date range: {master_df.index.min()} to {master_df.index.max()}")
print(f"Total days: {(master_df.index.max() - master_df.index.min()).days}")

# Define split date based on actual data range
# Use 85% of data for training, 15% for testing
total_days = (master_df.index.max() - master_df.index.min()).days
split_days = int(total_days * 0.85)
split_date = master_df.index.min() + timedelta(days=split_days)

print(f"\nUsing split date: {split_date}")

# Create train/test masks
train_mask = master_df.index < split_date
test_mask = master_df.index >= split_date

# Split baseline features
X_train_baseline = X_baseline[train_mask]
X_test_baseline = X_baseline[test_mask]

# Split advanced features
X_train_advanced = X_advanced[train_mask]
X_test_advanced = X_advanced[test_mask]

# Split target
y_train = y[train_mask]
y_test = y[test_mask]

print(f"Split date: {split_date}")
print(f"\nTraining set:")
print(f"  - Baseline: {X_train_baseline.shape}")
print(f"  - Advanced: {X_train_advanced.shape}")
print(f"  - Target: {y_train.shape}")
print(f"  - Date range: {X_train_baseline.index.min()} to {X_train_baseline.index.max()}")

print(f"\nTest set:")
print(f"  - Baseline: {X_test_baseline.shape}")
print(f"  - Advanced: {X_test_advanced.shape}")
print(f"  - Target: {y_test.shape}")
print(f"  - Date range: {X_test_baseline.index.min()} to {X_test_baseline.index.max()}")


### 5.3: Train Baseline Model (Linear Regression)


In [ ]:
print("Training baseline model (Linear Regression)...")
baseline_model = LinearRegression()
baseline_model.fit(X_train_baseline, y_train)

print("Baseline model trained successfully!")
print(f"Coefficients shape: {baseline_model.coef_.shape}")
print(f"Intercept: {baseline_model.intercept_:.2f}")


### 5.4: Train Advanced Model (XGBoost with News Embeddings)


In [ ]:
print("Training advanced model (XGBoost)...")
advanced_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=50,
    n_jobs=-1
)

# Train with early stopping using test set for validation
advanced_model.fit(
    X_train_advanced, 
    y_train,
    eval_set=[(X_test_advanced, y_test)],
    verbose=False
)

print("Advanced model trained successfully!")
print(f"Best iteration: {advanced_model.best_iteration}")
print(f"Best score: {advanced_model.best_score:.4f}")


## Module 6: Evaluation

### 6.1: Generate Predictions


In [ ]:
# Generate predictions on test set
y_pred_baseline = baseline_model.predict(X_test_baseline)
y_pred_advanced = advanced_model.predict(X_test_advanced)

print("Predictions generated successfully!")
print(f"Baseline predictions: {y_pred_baseline.shape}")
print(f"Advanced predictions: {y_pred_advanced.shape}")


### 6.2: Performance Metrics

Calculate MAE and RMSE for both models.


In [ ]:
# Calculate metrics for baseline model
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

# Calculate metrics for advanced model
advanced_mae = mean_absolute_error(y_test, y_pred_advanced)
advanced_rmse = np.sqrt(mean_squared_error(y_test, y_pred_advanced))

# Create comparison table
results_df = pd.DataFrame({
    'Model': ['Baseline (Linear Regression)', 'Advanced (XGBoost + News)'],
    'MAE (€/MWh)': [baseline_mae, advanced_mae],
    'RMSE (€/MWh)': [baseline_rmse, advanced_rmse]
})

# Calculate improvement
mae_improvement = ((baseline_mae - advanced_mae) / baseline_mae) * 100
rmse_improvement = ((baseline_rmse - advanced_rmse) / baseline_rmse) * 100

print("=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)
print(f"MAE Improvement: {mae_improvement:.2f}%")
print(f"RMSE Improvement: {rmse_improvement:.2f}%")
print("=" * 60)


### 6.3: Visualization - Actual vs Predicted Prices

Plot a sample week from the test set to visualize model performance.


In [ ]:
# Select one week from test set for visualization
sample_start = X_test_baseline.index.min()
sample_end = sample_start + timedelta(days=7)
sample_mask = (X_test_baseline.index >= sample_start) & (X_test_baseline.index < sample_end)

# Calculate residuals for the sample period
baseline_residuals_sample = y_test[sample_mask] - y_pred_baseline[sample_mask]
advanced_residuals_sample = y_test[sample_mask] - y_pred_advanced[sample_mask]

# Create figure with 2 subplots (stacked vertically)
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Top subplot: Actual vs Predicted Prices
axes[0].plot(
    X_test_baseline.index[sample_mask], 
    y_test[sample_mask], 
    label='Actual Price',
    linewidth=2,
    color='black'
)
axes[0].plot(
    X_test_baseline.index[sample_mask], 
    y_pred_baseline[sample_mask], 
    label='Baseline Model',
    linewidth=1.5,
    linestyle='--',
    alpha=0.8,
    color='#1f77b4'
)
axes[0].plot(
    X_test_baseline.index[sample_mask], 
    y_pred_advanced[sample_mask], 
    label='Advanced Model (with News)',
    linewidth=1.5,
    linestyle='-.',
    alpha=0.8,
    color='#ff7f0e'
)

axes[0].set_ylabel('Price (€/MWh)')
axes[0].set_title('Electricity Price Forecast: Actual vs Predicted (Sample Week)')
axes[0].legend(frameon=False, loc='best')
axes[0].grid(True, alpha=0.3)

# Bottom subplot: Residuals (Forecast Error)
axes[1].plot(
    X_test_baseline.index[sample_mask], 
    baseline_residuals_sample, 
    label='Baseline Model Error',
    linewidth=1.5,
    linestyle='--',
    alpha=0.8,
    color='#1f77b4'
)
axes[1].plot(
    X_test_baseline.index[sample_mask], 
    advanced_residuals_sample, 
    label='Advanced Model Error',
    linewidth=1.5,
    linestyle='-.',
    alpha=0.8,
    color='#ff7f0e'
)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

axes[1].set_xlabel('Date')
axes[1].set_ylabel('Forecast Error (€/MWh)')
axes[1].set_title('Forecast Error: Actual - Predicted (Positive = Underprediction)')
axes[1].legend(frameon=False, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Visualization period: {sample_start} to {sample_end}")
print(f"\nSample week error statistics:")
print(f"  Baseline - MAE: {np.abs(baseline_residuals_sample).mean():.2f} €/MWh, Std: {baseline_residuals_sample.std():.2f} €/MWh")
print(f"  Advanced - MAE: {np.abs(advanced_residuals_sample).mean():.2f} €/MWh, Std: {advanced_residuals_sample.std():.2f} €/MWh")


### 6.4: Feature Importance Analysis

Analyze which features are most important in the XGBoost model.


In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': advanced_features,
    'importance': advanced_model.feature_importances_
}).sort_values('importance', ascending=False)

# Identify which features are news embeddings
feature_importance['is_news_feature'] = feature_importance['feature'].str.startswith('news_embed_')

# Get top 20 features
top_features = feature_importance.head(20)

# Create visualization
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#d62728' if is_news else '#1f77b4' for is_news in top_features['is_news_feature']]

ax.barh(range(len(top_features)), top_features['importance'], color=colors, alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importance Score')
ax.set_title('Top 20 Most Important Features')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1f77b4', alpha=0.7, label='Baseline Features'),
    Patch(facecolor='#d62728', alpha=0.7, label='News Embedding Features')
]
ax.legend(handles=legend_elements, frameon=False)

plt.tight_layout()
plt.show()

# Summary statistics
news_features_in_top20 = top_features['is_news_feature'].sum()
total_news_importance = feature_importance[feature_importance['is_news_feature']]['importance'].sum()
total_baseline_importance = feature_importance[~feature_importance['is_news_feature']]['importance'].sum()

print(f"\nFeature Importance Summary:")
print(f"- News features in top 20: {news_features_in_top20}")
print(f"- Total news feature importance: {total_news_importance:.4f}")
print(f"- Total baseline feature importance: {total_baseline_importance:.4f}")
print(f"- News feature importance ratio: {total_news_importance / (total_news_importance + total_baseline_importance):.2%}")


### 6.5: Advanced Model Comparison Visualizations

Detailed visual analysis of model performance and prediction quality.


In [ ]:
# 1. Scatter Plots: Actual vs Predicted (both models)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline model
axes[0].scatter(y_test, y_pred_baseline, alpha=0.5, s=20, edgecolors='none')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'k--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (€/MWh)')
axes[0].set_ylabel('Predicted Price (€/MWh)')
axes[0].set_title(f'Baseline Model\nMAE: {baseline_mae:.2f} €/MWh')
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.3)

# Advanced model
axes[1].scatter(y_test, y_pred_advanced, alpha=0.5, s=20, edgecolors='none')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'k--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price (€/MWh)')
axes[1].set_ylabel('Predicted Price (€/MWh)')
axes[1].set_title(f'Advanced Model (XGBoost + News)\nMAE: {advanced_mae:.2f} €/MWh')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Scatter plots show how well predictions align with actual values.")
print("Points closer to the diagonal line indicate better predictions.")


In [ ]:
# 2. Residual Plots: Error vs Predicted Value
baseline_residuals = y_test - y_pred_baseline
advanced_residuals = y_test - y_pred_advanced

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline residuals
axes[0].scatter(y_pred_baseline, baseline_residuals, alpha=0.5, s=20, edgecolors='none')
axes[0].axhline(y=0, color='k', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Price (€/MWh)')
axes[0].set_ylabel('Residuals (€/MWh)')
axes[0].set_title('Baseline Model - Residual Plot')
axes[0].grid(True, alpha=0.3)

# Advanced residuals
axes[1].scatter(y_pred_advanced, advanced_residuals, alpha=0.5, s=20, edgecolors='none')
axes[1].axhline(y=0, color='k', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price (€/MWh)')
axes[1].set_ylabel('Residuals (€/MWh)')
axes[1].set_title('Advanced Model - Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Residual plots show prediction errors across different price levels.")
print("Good models have residuals randomly scattered around zero.")


In [ ]:
# 3. Error Distribution Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of errors
axes[0].hist(baseline_residuals, bins=30, alpha=0.6, label='Baseline', edgecolor='black')
axes[0].hist(advanced_residuals, bins=30, alpha=0.6, label='Advanced', edgecolor='black')
axes[0].axvline(x=0, color='k', linestyle='--', lw=2)
axes[0].set_xlabel('Prediction Error (€/MWh)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Error Distribution Comparison')
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot comparison
box_data = [baseline_residuals, advanced_residuals]
bp = axes[1].boxplot(box_data, labels=['Baseline', 'Advanced'], patch_artist=True)
axes[1].axhline(y=0, color='k', linestyle='--', lw=1, alpha=0.5)
axes[1].set_ylabel('Prediction Error (€/MWh)')
axes[1].set_title('Error Distribution - Box Plot')
axes[1].grid(True, alpha=0.3, axis='y')

# Color the boxes
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

plt.tight_layout()
plt.show()

# Calculate error statistics
print("Error Statistics:")
print(f"\nBaseline Model:")
print(f"  Mean error: {baseline_residuals.mean():.2f} €/MWh")
print(f"  Std dev: {baseline_residuals.std():.2f} €/MWh")
print(f"  Median error: {baseline_residuals.median():.2f} €/MWh")

print(f"\nAdvanced Model:")
print(f"  Mean error: {advanced_residuals.mean():.2f} €/MWh")
print(f"  Std dev: {advanced_residuals.std():.2f} €/MWh")
print(f"  Median error: {advanced_residuals.median():.2f} €/MWh")


In [ ]:
# 4. Time Series of Absolute Errors
baseline_abs_errors = np.abs(baseline_residuals)
advanced_abs_errors = np.abs(advanced_residuals)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(X_test_baseline.index, baseline_abs_errors, 
        label='Baseline', alpha=0.7, linewidth=1)
ax.plot(X_test_baseline.index, advanced_abs_errors, 
        label='Advanced', alpha=0.7, linewidth=1)

ax.set_xlabel('Date')
ax.set_ylabel('Absolute Error (€/MWh)')
ax.set_title('Prediction Errors Over Time')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Time series shows when each model performs well or poorly.")
print("Look for patterns related to specific dates or events.")


In [ ]:
# 5. Performance by Hour of Day
# Create a DataFrame for analysis
error_analysis = pd.DataFrame({
    'hour': X_test_baseline.index.hour,
    'baseline_abs_error': baseline_abs_errors,
    'advanced_abs_error': advanced_abs_errors
})

# Calculate mean absolute error by hour
hourly_performance = error_analysis.groupby('hour').mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Line plot
axes[0].plot(hourly_performance.index, hourly_performance['baseline_abs_error'], 
            marker='o', label='Baseline', linewidth=2)
axes[0].plot(hourly_performance.index, hourly_performance['advanced_abs_error'], 
            marker='s', label='Advanced', linewidth=2)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Absolute Error (€/MWh)')
axes[0].set_title('Model Performance by Hour of Day')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.3)

# Improvement/degradation by hour
hourly_diff = hourly_performance['baseline_abs_error'] - hourly_performance['advanced_abs_error']
colors = ['green' if x > 0 else 'red' for x in hourly_diff]

axes[1].bar(hourly_performance.index, hourly_diff, color=colors, alpha=0.6, edgecolor='black')
axes[1].axhline(y=0, color='k', linestyle='--', lw=1)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Error Difference (€/MWh)')
axes[1].set_title('Performance Difference: Baseline - Advanced\n(Positive = Advanced is Better)')
axes[1].set_xticks(range(0, 24, 2))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Find best and worst performing hours for each model
best_hour_baseline = hourly_performance['baseline_abs_error'].idxmin()
worst_hour_baseline = hourly_performance['baseline_abs_error'].idxmax()
best_hour_advanced = hourly_performance['advanced_abs_error'].idxmin()
worst_hour_advanced = hourly_performance['advanced_abs_error'].idxmax()

print("Performance by Hour of Day:")
print(f"\nBaseline Model:")
print(f"  Best hour: {best_hour_baseline}:00 (MAE: {hourly_performance.loc[best_hour_baseline, 'baseline_abs_error']:.2f} €/MWh)")
print(f"  Worst hour: {worst_hour_baseline}:00 (MAE: {hourly_performance.loc[worst_hour_baseline, 'baseline_abs_error']:.2f} €/MWh)")

print(f"\nAdvanced Model:")
print(f"  Best hour: {best_hour_advanced}:00 (MAE: {hourly_performance.loc[best_hour_advanced, 'advanced_abs_error']:.2f} €/MWh)")
print(f"  Worst hour: {worst_hour_advanced}:00 (MAE: {hourly_performance.loc[worst_hour_advanced, 'advanced_abs_error']:.2f} €/MWh)")

print(f"\nHours where Advanced model outperforms Baseline: {(hourly_diff > 0).sum()}/24")


In [ ]:
# 6. Comprehensive Metrics Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Metric 1: MAE and RMSE comparison
metrics = ['MAE', 'RMSE']
baseline_metrics = [baseline_mae, baseline_rmse]
advanced_metrics = [advanced_mae, advanced_rmse]

x = np.arange(len(metrics))
width = 0.35

axes[0, 0].bar(x - width/2, baseline_metrics, width, label='Baseline', alpha=0.7, edgecolor='black')
axes[0, 0].bar(x + width/2, advanced_metrics, width, label='Advanced', alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Error (€/MWh)')
axes[0, 0].set_title('Model Performance Metrics')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend(frameon=False)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Metric 2: R² Score
from sklearn.metrics import r2_score
baseline_r2 = r2_score(y_test, y_pred_baseline)
advanced_r2 = r2_score(y_test, y_pred_advanced)

axes[0, 1].bar(['Baseline', 'Advanced'], [baseline_r2, advanced_r2], 
               alpha=0.7, edgecolor='black')
axes[0, 1].set_ylabel('R² Score')
axes[0, 1].set_title('Coefficient of Determination (R²)')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, (model, r2) in enumerate(zip(['Baseline', 'Advanced'], [baseline_r2, advanced_r2])):
    axes[0, 1].text(i, r2 + 0.02, f'{r2:.3f}', ha='center', va='bottom', fontweight='bold')

# Metric 3: Prediction Accuracy within thresholds
thresholds = [5, 10, 15, 20, 25]
baseline_within = [(np.abs(baseline_residuals) <= t).mean() * 100 for t in thresholds]
advanced_within = [(np.abs(advanced_residuals) <= t).mean() * 100 for t in thresholds]

axes[1, 0].plot(thresholds, baseline_within, marker='o', label='Baseline', linewidth=2)
axes[1, 0].plot(thresholds, advanced_within, marker='s', label='Advanced', linewidth=2)
axes[1, 0].set_xlabel('Error Threshold (€/MWh)')
axes[1, 0].set_ylabel('Predictions Within Threshold (%)')
axes[1, 0].set_title('Prediction Accuracy at Different Thresholds')
axes[1, 0].legend(frameon=False)
axes[1, 0].grid(True, alpha=0.3)

# Metric 4: Mean Absolute Percentage Error
baseline_mape = (np.abs(baseline_residuals) / np.abs(y_test)).mean() * 100
advanced_mape = (np.abs(advanced_residuals) / np.abs(y_test)).mean() * 100

axes[1, 1].bar(['Baseline', 'Advanced'], [baseline_mape, advanced_mape], 
               alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('MAPE (%)')
axes[1, 1].set_title('Mean Absolute Percentage Error')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, (model, mape) in enumerate(zip(['Baseline', 'Advanced'], [baseline_mape, advanced_mape])):
    axes[1, 1].text(i, mape + 0.5, f'{mape:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print comprehensive summary
print("\n" + "="*70)
print("COMPREHENSIVE MODEL COMPARISON SUMMARY")
print("="*70)
print(f"\n{'Metric':<40} {'Baseline':<15} {'Advanced':<15}")
print("-"*70)
print(f"{'Mean Absolute Error (MAE)':<40} {baseline_mae:>10.2f} €/MWh {advanced_mae:>10.2f} €/MWh")
print(f"{'Root Mean Squared Error (RMSE)':<40} {baseline_rmse:>10.2f} €/MWh {advanced_rmse:>10.2f} €/MWh")
print(f"{'R² Score':<40} {baseline_r2:>14.3f} {advanced_r2:>14.3f}")
print(f"{'Mean Absolute Percentage Error (MAPE)':<40} {baseline_mape:>11.2f} % {advanced_mape:>11.2f} %")
print(f"{'Predictions within ±10 €/MWh':<40} {baseline_within[1]:>11.1f} % {advanced_within[1]:>11.1f} %")
print(f"{'Predictions within ±20 €/MWh':<40} {baseline_within[3]:>11.1f} % {advanced_within[3]:>11.1f} %")
print("="*70)
